# L18 · 可逆性与秩（rank） — 秩 = 信息量，零空间（null space） = 被压缩的方向，奇异矩阵诊断

**目标**：判断一个方阵是否可逆的三条判据，以及「严格对角占优」这一快速充分条件。

**为什么对 Aurora 重要**：Jacobi/Gauss-Seidel 迭代求解器以对角占优为收敛前提；`is_sdd` 检查是求解器入口的第一道关卡。

← **上一课**　[L17 · 特征分解 A=PDP⁻¹](L17_eigen_diagonalization.ipynb)

> 上节课学习了 **特征分解 A=PDP⁻¹**：换坐标系让矩阵乘法变成标量乘法。  
> 本课将探讨 **可逆性与秩**。

## 本课剧情：判断过程能不能倒带

可逆性问一个朴素问题：看到输出以后，我们还能不能找回唯一输入？`det(A) ≠ 0` 和「所有特征值（eigenvalue）非零」是两条充要条件，严格对角占优是不用算特征值就能快速确认可逆的充分条件。

## 1. 三条判据

1. **行列式（determinant） ≠ 0**（充要）
2. **所有特征值 ≠ 0**（充要）
3. **严格对角占优 (s.d.d.)**：每行对角元的绝对值 > 该行其余元素绝对值之和（**充分**但非必要）。

三条判据的实用定位不同：`det ≠ 0` 和「所有特征值 ≠ 0」是充要条件，两者等价，但计算代价都是 O(n³)（LU 分解 / QR 迭代）。严格对角占优只是充分条件——不满足它矩阵仍可能可逆——但检查代价仅 O(n²)，且它直接保证 Jacobi/Gauss-Seidel 迭代的谱半径（spectral radius） < 1，即迭代误差必然收敛。实际工程策略：先用 `is_sdd` 快速筛查；通过则启动迭代求解器；不通过再按需计算 det 或特征值。

## 符号入口：先看形状，再看运算

本节处理方阵 `A`（shape `(n, n)`），核心工具是 `np.linalg.det(A)` 和 `np.linalg.eigvals(A)`。判断可逆性归结为两个数字：一个行列式标量和一组特征值向量（vector）。

In [ ]:
import numpy as np
A = np.array([[1,1,0],[1,2,1],[0,1,3]], float)
print('det =', round(np.linalg.det(A)))
print('特征值:', np.round(np.linalg.eigvals(A), 3))
print('两条充要判据都说：可逆 =', abs(np.linalg.det(A))>1e-9)

## 动手观察：从 det 和特征值读可逆性

运行下面的代码，注意 `det` 接近零时特征值中是否出现接近零的分量——两条充要判据在数值上指向同一个事实。

In [ ]:
import numpy as np

# 秩 = 线性无关行（列）的数目
mats = [
    ('满秩', np.array([[1.,0.,0.],[0.,1.,0.],[0.,0.,1.]])),
    ('秩2', np.array([[1.,2.,3.],[4.,5.,6.],[7.,8.,9.]])),
    ('秩1', np.array([[1.,2.],[2.,4.]])),
]
for name, A in mats:
    r = np.linalg.matrix_rank(A)
    inv_ok = (r == min(A.shape))
    print(f'{name}  shape={A.shape}  rank={r}  可逆={inv_ok}')


## 代码实验：观察矩阵如何作用于多个向量

可逆矩阵把不同的输入映射到不同的输出；奇异矩阵则会把两个不同向量压缩到同一个方向。

In [ ]:
import numpy as np

# 零空间（核）= SVD 中对应零奇异值的右奇异向量
A = np.array([[1.,2.,3.],[4.,5.,6.],[7.,8.,9.]])
U, s, Vt = np.linalg.svd(A)
print(f'奇异值 σ = {np.round(s, 6)}')
# 零奇异值对应 Vt 最后一行
null_vec = Vt[-1]
print(f'零空间向量 n = {np.round(null_vec, 4)}')
print(f'A @ n = {np.round(A @ null_vec, 10)}  （应为全零）')


## ✏️ 练习：提取零空间向量并验证

奇异矩阵的零空间（null space）不止是「存在」，还可以用 SVD 精确提取。

**任务**：给定秩亏矩阵 `B`，找到零空间向量 `v`，使得 `B @ v ≈ 0`。

**推理路线**：
1. 对 `B` 做 SVD：`U, s, Vt = np.linalg.svd(B)`
2. 最小奇异值对应 `Vt` 的最后一行，即零空间向量。
3. 如果该奇异值 < 1e-9，则 `B` 实际上秩亏，`Vt[-1]` 是零空间向量。
4. 验证：`‖B @ v‖ < 1e-8`。

In [ ]:
import numpy as np

def extract_null_vector(B):
    """返回矩阵 B 的（最小奇异值对应）零空间向量 v。
    若 B 是满秩矩阵，返回最接近零空间的向量（最小奇异值对应方向）。
    """
    B = np.asarray(B, float)
    # ✏️ TODO: 用 SVD 提取零空间向量
    # 提示：np.linalg.svd(B) 返回 U, s, Vt；
    #       最小奇异值对应 Vt 的最后一行。
    raise NotImplementedError("TODO: implement extract_null_vector — 用 np.linalg.svd(B) 提取 Vt[-1]")

# 测试：秩亏矩阵（rank 2，有真实零空间）
B = np.array([[1., 2., 3.],
              [4., 5., 6.],
              [7., 8., 9.]])

try:
    v = extract_null_vector(B)
    residual = np.linalg.norm(B @ v)
    print(f'零空间向量 v = {np.round(v, 6)}')
    print(f'‖B @ v‖ = {residual:.2e}  （应 < 1e-8，即数值零）')
    assert residual < 1e-8, f'残差过大：{residual:.2e}，请检查实现'
    print('✅ 零空间向量验证通过！')
except NotImplementedError as e:
    print(f'⚠️  {e}')
    print('请先完成 extract_null_vector 的 TODO 实现，再运行此单元格。')


## 2. ✏️ 实现 `is_sdd(A)`（严格对角占优）

对每行 i：`|A[i,i]| > Σ_{j≠i} |A[i,j]|` 是否都成立。

**推理路线**：
1. 取第 i 行的对角元绝对值：`d = abs(A[i, i])`。
2. 取第 i 行所有元素绝对值之和：`s = sum(abs(A[i]))`；非对角部分之和为 `s - d`（无需单独遍历 j≠i）。
3. 检查 `d > s - d` 对所有行成立；用 `all(...)` 把 n 个布尔值合并为一个返回值。

**参考输入输出**：`A=[[4,1],[1,3]]` → 行0：4>1 ✓；行1：3>1 ✓ → `True`；`B=[[1,2],[2,1]]` → 行0：1<2 ✗ → `False`

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写 `is_sdd` 前明确三件事

- 输入：`A`，shape `(n, n)` 的方阵
- 关键步骤：对每行 `i`，比较 `abs(A[i,i])` 与 `sum(abs(A[i])) - abs(A[i,i])`
- 返回：`bool`，所有行均满足则为 `True`

In [ ]:
def is_sdd(A):
    A = np.asarray(A, float)
    # ✏️ TODO: 返回是否严格对角占优 (True/False)
    raise NotImplementedError("TODO: implement is_sdd — 参考推理路线：d = abs(A[i,i])，s = sum(abs(A[i]))，检查 d > s - d")


In [ ]:
# 三条可逆性判据验证例题
M1 = np.array([[2,0,1],[1,4,2],[1,3,6]], float)   # s.d.d. 且可逆
M2 = np.array([[1,0,2],[2,5,1],[3,2,13]], float)  # 非 s.d.d. 但仍可逆
M3 = np.array([[1,1],[1,1]], float)               # 非 s.d.d. 且不可逆
for name, M in [('M1',M1),('M2',M2),('M3',M3)]:
    print(f'{name}: sdd={is_sdd(M)}, 可逆={abs(np.linalg.det(M))>1e-9}')
try:
    assert is_sdd(M1) and not is_sdd(M2) and not is_sdd(M3)
    print('\n✅ 与课件三例一致。注意 M2 说明：sdd 是充分非必要条件。')
except NotImplementedError as e:
    print(f'⚠️  {e}')
    print('请先完成上方 is_sdd 的 TODO 实现，再运行此验证单元格。')

**🔗 Aurora 连接**：`is_sdd` 是 Aurora 迭代求解器的入口卫士——系数矩阵只有通过 s.d.d. 检查，Jacobi/Gauss-Seidel 求解流程才会启动；此时谱半径 < 1 有数学保证，迭代误差按几何级数衰减而非发散。

**补充例题对应**：三条可逆性判据 + 严格对角占优。

**线代前导章节完成**：L09–L18 覆盖向量、矩阵、方程组、行列式、特征值与可逆性。

## 🎨 图示：严格对角占优矩阵(对角线显著占优 ⇒ 可逆)

In [ ]:
from aurora.laviz import style, heatmap
style()
heatmap([[2,0,1],[1,4,2],[1,3,6]], title='严格对角占优 ⇒ 可逆', cmap='magma');

In [ ]:
import numpy as np

# 参数实验：矩阵退化实验——让一列线性依赖于其他列
print(f"{'依赖系数 c':>12}  {'rank':>6}  {'det':>12}  {'可逆':>6}")
for c in [0.0, 0.5, 1.0, 1.5, 2.0]:
    A = np.array([[1.,2.,c],[0.,1.,1.],[0.,0.,c-1.]])
    r = np.linalg.matrix_rank(A)
    d = np.linalg.det(A)
    print(f'{c:>12.1f}  {r:>6d}  {d:>+12.4f}  {"是" if r==3 else "否":>6s}')


## 参数实验：Jacobi 迭代的收敛与发散

构造一个对角元明显大于同行其余元素绝对值之和的矩阵（如 `A = [[10,1,1],[1,10,1],[1,1,10]]`），用 Jacobi 迭代求解线性系统 `Ax = b`，打印每轮残差 `‖Ax_k - b‖`，确认误差逐步缩小至收敛。

再把对角元缩小使矩阵不再满足 s.d.d.（如 `A = [[1,2,2],[2,1,2],[2,2,1]]`），重跑相同迭代步数，观察残差不减反增或振荡。两次实验对比说明：`is_sdd` 通过是 Jacobi 收敛的理论保证，不是可有可无的布尔标记。

In [ ]:
import numpy as np

def jacobi(A, b, n_iter=30):
    """Jacobi 迭代：x_new[i] = (b[i] - Σ_{j≠i} A[i,j]·x[j]) / A[i,i]"""
    x = np.zeros_like(b, dtype=float)
    D_inv = 1.0 / np.diag(A)           # 对角元倒数
    residuals = []
    for _ in range(n_iter):
        x = D_inv * (b - (A @ x - np.diag(np.diag(A)) @ x))
        residuals.append(np.linalg.norm(A @ x - b))
    return x, residuals

# ① s.d.d. 矩阵 → 收敛
A_sdd = np.array([[10.,1.,1.],[1.,10.,1.],[1.,1.,10.]])
b_sdd = np.array([12.,12.,12.])
x_sdd, res_sdd = jacobi(A_sdd, b_sdd)
print('s.d.d. 最终残差:', f'{res_sdd[-1]:.2e}  (应趋近 0)')
assert res_sdd[-1] < 1e-8, 'Jacobi 在 s.d.d. 矩阵上应收敛'

# ② 非 s.d.d. 矩阵 → 发散
A_bad = np.array([[1.,2.,2.],[2.,1.,2.],[2.,2.,1.]])
b_bad = np.ones(3)
_, res_bad = jacobi(A_bad, b_bad)
print('非 s.d.d. 最终残差:', f'{res_bad[-1]:.2e}  (期望 >> 0，发散)')
print('✅ Jacobi 收敛 vs 发散对比完成')

## 本课收束

现在能用 `is_sdd(A)` 逐行检查严格对角占优，用 `np.linalg.det` 和 `np.linalg.eigvals` 验证两条充要判据。`is_sdd` 对应 Aurora 迭代求解器的入口检查：通过它，Jacobi 迭代才能保证收敛。下一课：**L19** 矩阵变换可视化——把线性变换的几何效果画出来，直观理解矩阵对空间的作用。

---

→ **下一课**　[L19 · 矩阵变换图解](L19_visual_multiply.ipynb)

> 下节课将学习 **矩阵变换图解**：旋转、缩放、剪切的动态视觉演示。